In [1]:
import numpy as np
import pandas as pd
import scipy

from IPython.display import clear_output

import sys
sys.path.append('../../../../Documents/GitHub/gustav/src/')

from gustav import ebi, ncbi, nlm, biogrid, nih, openalex
from gustav import publications
from gustav import github
from gustav import access_framework
from gustav import mapper

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import auc
from scipy.stats import fisher_exact
pd.options.display.precision = 3
pd.options.display.expand_frame_repr = False
pd.options.display.max_columns = 20

import gc
import os
import json
import re

In [2]:
import cv2
import pytesseract

import re
from pathlib import Path
import requests
import random
import time
import scipy
from bs4 import BeautifulSoup

pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

In [3]:
UserAgent = [
        "Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.1 (KHTML, like Gecko) Chrome/22.0.1207.1 Safari/537.1"
        ]


def requestHeader(url):
    # Build request headers
    headers = {
            'User-Agent':random.choice(UserAgent),
            'Referer': url,
            'Connection':'keep-alive'
            }
    return headers

# Check for Tauc in articles for which full text has already been downloaded

In [11]:
sem_df = pd.read_csv('../data/240107_combined_sem_labels.csv')

In [ ]:
tauc_dois = []
for doi in sem_df['doi']:
    with open('../data/sem_full_text/' + doi.replace('/', '_') + '.txt', encoding='latin') as f:
        full_str = f.read()
    if 'tauc' in full_str.lower():
        tauc_dois.append(doi)

sem_df[sem_df['doi'].isin(tauc_dois)].to_csv('../data/241127_sem_images_tauc_reexam.csv', index=False)

In [2]:
os.listdir('../data/')

['240107_combined_sem_labels.csv',
 '240107_combined_sem_metadata.csv',
 '240107_combined_sem_works.csv',
 '240829_works_additive_manufacturing.csv',
 '240903_retraction_watch_db.csv',
 'full_text_sem_images_addma_240830.csv',
 'sem_full_text',
 'sem_images_240829_batch.csv',
 'sem_images_240829_batch_count_1000_.csv',
 'sem_images_240829_batch_count_2000_.csv',
 'sem_images_240829_batch_count_3000_.csv',
 'sem_images_240829_batch_count_3057_.csv',
 'tesseract_ocr_sem_batch_240829_addma.csv',
 'tesseract_ocr_sem_batch_240829_addma_1000.csv',
 'tesseract_ocr_sem_batch_240829_addma_2000.csv',
 'tesseract_ocr_sem_batch_240829_addma_2316.csv']

# Check for Tauc in a sample of articles for which full text has not yet been download (SEM images but not metadata)
Go until there are ~300 Tauc images

In [4]:
sem_df = pd.read_csv('../data/240107_combined_sem_metadata.csv')
labels_df = pd.read_csv('../data/240107_combined_sem_labels.csv')

C:\Users\richa\AppData\Local\Temp\ipykernel_7116\767287462.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  sem_df = pd.read_csv('../data/240107_combined_sem_metadata.csv')


In [5]:
meta_df = sem_df[~sem_df['doi'].isin(labels_df['doi'])].copy()

In [6]:
def frontiers_full_text(doi):
    url = 'https://doi.org/' + doi

    response = requests.get(url, headers= requestHeader(url))
        
    file = BeautifulSoup(response.text, "lxml")
            
    full_text = ' '.join([p.text for p in file.find_all('p')])
            
    file.decompose()
    
    return full_text
    
def plos_full_text(doi):
    url = 'https://doi.org/' + doi

    response = requests.get(url, headers= requestHeader(url))
        
    file = BeautifulSoup(response.text, "lxml")
            
    full_text = ' '.join([p.text for p in file.find_all('p')])
            
    file.decompose()
    
    return full_text
    
def nature_full_text(doi):
    
    url = 'https://doi.org/' + doi

    response = requests.get(url, headers= requestHeader(url))
        
    file = BeautifulSoup(response.text, "lxml")
            
    full_text = ' '.join([p.text for p in file.find_all('p')])
            
    file.decompose()
    
    return full_text
    
def elsevier_full_text(doi, api_key):
    
    url = 'https://api.elsevier.com/content/article/doi/' + doi + '?httpAccept=text/plain' + '&APIKey=' + api_key
    
    response = requests.get(url, headers= requestHeader(url))
    
    return response.text

In [7]:
np.random.seed(50)
meta_df = meta_df.drop_duplicates('doi').sample(frac=1).reset_index(drop=True)

In [8]:
with open('../secret/elsevier_api.txt') as f:
    api_key = f.read()

In [9]:
len(meta_df)-meta_df.loc[meta_df['doi'].isin(['10.1016/j.conbuildmat.2021.125120'])].index.values[0]

50451

In [ ]:
elsevier_full_text('10.1016/j.cej.2021.129736', api_key)

In [19]:
remaining_df = pd.read_csv('../data/full_text_timeout.txt', header=None)
remaining_df.columns = ['doi']

In [18]:
for index, row in meta_df.iterrows():
    doi = row['doi']
    publisher = doi.split('/')[0]

    try:
        if publisher == '10.1016':
            full_text = elsevier_full_text(doi, api_key)
        elif publisher == '10.1371':
            full_text = plos_full_text(doi)
        elif publisher == '10.1038':
            full_text = nature_full_text(doi)
        elif publisher == '10.3389':
            full_text = frontiers_full_text(doi)
            
        filename = '../data/sem_full_text/' + doi.replace('/','_') + '.txt'

        with open(filename, 'w', encoding='utf') as f:
            f.write(full_text)
            
    except:
        print('Error processing DOI: ' + doi)
        with open('../data/full_text_timeout.txt', 'a+') as f:
            f.write(doi + '\n')

    print(str(index) + ' DOIs processed.')
        
    if index % 10 == 0:
        clear_output()
        gc.collect()

101 DOIs processed.
102 DOIs processed.
103 DOIs processed.
104 DOIs processed.
105 DOIs processed.
106 DOIs processed.
107 DOIs processed.
108 DOIs processed.


In [21]:
tauc_dois = []
for doi in meta_df['doi']:
    with open('../data/sem_full_text/' + doi.replace('/', '_') + '.txt', encoding='latin') as f:
        full_str = f.read()
    if 'tauc' in full_str.lower():
        tauc_dois.append(doi)

In [22]:
len(tauc_dois)

6108

In [32]:
meta_df_slice = meta_df[meta_df['doi'].isin(tauc_dois)].head(500)

In [33]:
meta_df_slice['doi_link'] = meta_df_slice['doi'].apply(lambda x: 'https://doi.org/' + x)

In [34]:
meta_df_slice[['doi_link']].to_csv('../data/241206_sem_images_tauc_reexam_negatives.csv', index=False)